# 01 - Spatiotemporal cluster update

Thin caller over `nfip.st_clustering`.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))   # the nfip package
sys.path.insert(0, os.path.abspath('src'))       # existing ST_Cluster.py, if used
import numpy as np, pandas as pd
from nfip import data, preprocess, config
from nfip.st_clustering import (
    split_by_temporal_gaps, cluster_temporal_by_county_for_splits,
    add_split_id_to_splits, make_temporal_global_ids,
    build_proxy_points_for_splits_cluster_only, run_stdbscan_on_proxy_splits,
    ensure_proxy_seed_labels, compute_st1st_map_for_split,
    prepare_proxy_points_with_st1st, apply_st1st_to_split_claims,
    attach_unclustered_to_st1st_for_split, make_global_st_ids,
    assemble_cols, print_cluster_totals_and_neg1,
    collect_temporal_cluster_lengths, plot_cluster_length_pdf,
)
# Provided by existing src/ST_Cluster.py:
from ST_Cluster import sensitivity_analysis

## Clustering params

In [ ]:
space_thres_list = [3]
time_thres_list = [5]
num_thres_list = [7]

temp_folder = 'Clusters/2025_all'
os.makedirs(temp_folder, exist_ok=True)
damage_only = True
save = False
save_small = False

## Load + preprocess claims

In [ ]:
claims = data.load_raw_claims()
claims = preprocess.preprocess_raw_claims(claims, damage_only=damage_only)

## Temporal DBSCAN within quiet-period splits

In [ ]:
time_thres = time_thres_list[0]
splits, gap_count, gap_points, sizes = split_by_temporal_gaps(claims, time_thres)
print(f'Total quiet-period gaps > {time_thres} days: {gap_count}')
for i, (gap, size) in enumerate(zip(gap_points, sizes[:-1]), start=1):
    print(f'Split {i}: {size} rows, ends before gap at day {gap}')
print(f'Final split {len(splits)}: {sizes[-1]} rows')

In [ ]:
time_thres = time_thres_list[0]
num_thres  = num_thres_list[0]
split_results, split_stats = cluster_temporal_by_county_for_splits(
    splits, time_thres=time_thres, num_thres=num_thres, label_col='temporal_cluster')
print('\nPer-split summary:')
print(split_stats.to_string(index=False))

In [ ]:
split_results = add_split_id_to_splits(split_results)
split_results = make_temporal_global_ids(
    split_results, county_col='countyCode',
    temp_col='temporal_cluster', new_col='temporal_cluster_gid')

## Proxy points + ST-DBSCAN

In [ ]:
proxy_frames = build_proxy_points_for_splits_cluster_only(
    split_results, time_thres=time_thres, label_col='temporal_cluster_gid', date_col='date')

In [ ]:
space_thres_list = [3]
time_thres_list  = [5]
num_thres_list   = [1]  # already clustered
proxy_frames, run_summary, run_totals = run_stdbscan_on_proxy_splits(
    proxy_frames, space_thres_list=space_thres_list,
    time_thres_list=time_thres_list, num_thres_list=num_thres_list,
    lat_col='latitude', lon_col='longitude')
print('\nPer-split results:'); print(run_summary.to_string(index=False))
print('\nTotals across splits:'); print(run_totals.to_string(index=False))

In [ ]:
proxy_frames = ensure_proxy_seed_labels(
    proxy_frames, space_thres=space_thres_list[0], time_thres=time_thres_list[0],
    min_samples=num_thres_list[0], county_col='countyCode',
    temp_col='original_temporal_cluster',
    st_col=f'st_cluster_{space_thres_list[0]}_{time_thres_list[0]}_{num_thres_list[0]}',
    out_col='st_seed', make_global=False)

## First-pass ST labels, attach unclustered, globalize

In [ ]:
space_thres = space_thres_list[0]
time_thres  = time_thres_list[0]
min_samples = num_thres_list[0]
st1st_maps      = [compute_st1st_map_for_split(p, space_thres, time_thres, min_samples) for p in proxy_frames]
proxies_with_s1 = [prepare_proxy_points_with_st1st(p, m) for p, m in zip(proxy_frames, st1st_maps)]
splits_with_s1  = [apply_st1st_to_split_claims(s, m) for s, m in zip(split_results, st1st_maps)]

In [ ]:
splits_final = [attach_unclustered_to_st1st_for_split(
                    s, p, space_thres, time_thres,
                    lat_col='latitude', lon_col='longitude', day_col='daysSinceStart',
                    temp_gid_col='temporal_cluster_gid')
                for s, p in zip(splits_with_s1, proxies_with_s1)]

In [ ]:
splits_final = make_global_st_ids(splits_final, st_col='st_cluster_1st',  new_col='st_cluster_1st_gid')
splits_final = make_global_st_ids(splits_final, st_col='st_cluster_final', new_col='st_cluster_final_gid')

temp_map = assemble_cols(splits_final, ['index','temporal_cluster','temporal_cluster_gid'])
st1_map  = assemble_cols(splits_final, ['index','st_cluster_1st','st_cluster_1st_gid'])
stf_map  = assemble_cols(splits_final, ['index','st_cluster_final','st_cluster_final_gid'])

claims_with_clusters = (claims
    .merge(temp_map, on='index', how='left')
    .merge(st1_map,  on='index', how='left')
    .merge(stf_map,  on='index', how='left'))

for c in ['temporal_cluster','temporal_cluster_gid',
          'st_cluster_1st','st_cluster_1st_gid',
          'st_cluster_final','st_cluster_final_gid']:
    claims_with_clusters[c] = claims_with_clusters[c].fillna(-1).astype('int32')

In [ ]:
print_cluster_totals_and_neg1(claims_with_clusters)

In [ ]:
if save:
    claims_with_clusters.to_csv('new_clusters_9.24.25.csv', index=False)
if save_small:
    claims_with_clusters[['id','dateOfLoss','longitude','latitude','temporal_cluster_gid','st_cluster_final_gid']].to_csv('cluster_update_9.24.25.csv', index=False)

In [ ]:
lengths = collect_temporal_cluster_lengths(split_results, label_col='temporal_cluster')
if not lengths.empty:
    print('Temporal cluster length stats (days):', {
        'count': int(lengths.size), 'min': int(lengths.min()),
        'p05': float(np.percentile(lengths, 5)), 'median': float(np.median(lengths)),
        'p95': float(np.percentile(lengths, 95)), 'max': int(lengths.max()),
        'mean': float(lengths.mean())})
plot_cluster_length_pdf(lengths, bins='auto')

## County analysis (uses existing `ST_Cluster.sensitivity_analysis`)

In [ ]:
claims, sensitivities = sensitivity_analysis(claims, space_thres_list, time_thres_list, num_thres_list)
if save:
    sensitivities.to_csv(f'{temp_folder}/cluster_sensitivities_cl.csv', index=False)
    claims.to_csv(f'{temp_folder}/clustered_claims_sensitivity.csv', index=False)
if save_small:
    claims[['id','dateOfLoss','longitude','latitude','st_cluster_3_5_7']].to_csv(f'{temp_folder}/clustered_claims_export.csv', index=False)